# Sinhala QA Evaluation — Google Colab (v6-matched)

Colab port of `llama-scripts/qa-evaluation.ipynb` (which runs on Modal). The evaluation
logic below — prompt construction, evidence-window retrieval, the grounding gate, and all
metrics — is **byte-for-byte identical** to the Modal notebook, so results from the two are
directly comparable. Only the platform glue changed: package install, GPU/runtime check,
data loading, and Hugging Face credentials now use Colab's own conventions instead of
Modal's `/tmp` filesystem and secrets.

**This notebook must reproduce the exact prompt format the model was fine-tuned on**
(`qa-finetuning_v6.ipynb`): the instruction text, the `සන්දර්භය:` / `ප්‍රශ්නය:` labels, and
critically the answer cue `පිළිතුර:\n` (a newline, **not** `[`). An earlier version of the
Modal notebook used a different prompt the model had never been trained on and scored far
below the true result for reasons unrelated to model quality — the smoke-test cell below
asserts the format on every run so that mistake can't silently recur.

## Before running

1. **Runtime > Change runtime type > T4 GPU** (or better) — CPU inference here would take
   hours. Free-tier Colab's T4 does not support bf16 acceleration; the notebook detects
   this and falls back to fp16 automatically.
2. **Provide `test.jsonl`**, in any of these ways — the loading cell tries all of them in
   order and does not fail until none work:
   - Upload it directly in the Colab file browser (left sidebar) to `/content/`.
   - Mount Google Drive and place it anywhere under `/content/drive/MyDrive/`.
   - Run the loading cell with nothing prepared: it will prompt an interactive file-upload
     dialog automatically.
3. **If the model repo is private**, add a Colab secret: click the key icon in the left
   sidebar, add a secret named `HF_TOKEN` with your Hugging Face token, enable notebook
   access. Never paste a token directly into a cell — a token pasted into a notebook is a
   leaked credential the moment the notebook is shared or committed.

In [ ]:
%pip install -q "transformers>=4.51,<5" accelerate safetensors huggingface_hub hf_transfer "scikit-learn>=1.5"
#
# The scikit-learn pin is the one that matters on hosted Jupyter platforms with a pre-baked
# conda/micromamba base environment (Vertex AI Workbench, Kaggle, some managed images).
# `transformers.generation.utils` optionally imports sklearn for an assisted-decoding
# feature this notebook never uses (plain greedy `model.generate()` doesn't touch it), via
# `transformers/generation/candidate_generator.py`. If the base image's scikit-learn
# predates NumPy 2.0 support, that import chain fails with:
#   ImportError: cannot import name 'ComplexWarning' from 'numpy.core.numeric'
# which then surfaces one layer up as the more confusing
#   ImportError: cannot import name 'GenerationMixin' from 'transformers.generation'
# This is a NumPy-2.0/scikit-learn version mismatch, not a transformers/tokenizers
# mismatch — scikit-learn 1.5+ added NumPy 2.0 support, and upgrading it (not numpy itself,
# which is riskier to touch in a shared base image) resolves it directly.
#
# If you still hit an ImportError after this: **restart the kernel/runtime** (Runtime/Kernel
# > Restart) before re-running from the top — a pip upgrade never takes effect in a Python
# process that already imported the old module.

In [ ]:
import os

import torch
import transformers

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("transformers  :", transformers.__version__, "->", transformers.__file__)
try:
    from transformers.generation import GenerationMixin  # noqa: F401
    print("GenerationMixin import: OK")
except ImportError as error:
    raise ImportError(
        "transformers is still broken after the install cell above. Restart the "
        "kernel/runtime (this is required — a pip upgrade does not apply to an already-"
        "running Python process) and re-run this notebook from the top. If it persists, "
        "the base environment may be pinning an incompatible tokenizers/accelerate build."
    ) from error

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"GPU              : {name}")
    print(f"BF16 accelerated : {bf16_ok}  "
          f"({'will use bf16' if bf16_ok else 'will use fp16 — normal on T4/free-tier Colab'})")
else:
    print("WARNING: no GPU detected.")
    print("Go to Runtime > Change runtime type and select a GPU (T4 is available on the free")
    print("tier). Evaluation will still run on CPU but generation will take a long time.")

In [ ]:
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

MODEL_ID = "isji/sinllama-3b-qa-v6-merged"   # or "isji/sinllama-1b-qa-v6-merged"

# ---- Locate test.jsonl: try common Colab locations, then fall back to an upload prompt ----
TEST_CANDIDATES = [
    Path("/content/test_updated.jsonl"),
    Path("/content/test.jsonl"),
    Path("/content/drive/MyDrive/test.jsonl"),
    Path("/content/drive/MyDrive/sinhala-qa/test.jsonl"),
    Path("/content/drive/MyDrive/test_updated.jsonl"),
    Path("/home/jupyter/tmp/test.jsonl"),          # Vertex AI Workbench-style environments
    Path("/home/jupyter/tmp/test_updated.jsonl"),
    Path("/home/jupyter/test.jsonl"),
    Path("new_split_v2/test.jsonl"),   # if this notebook sits inside the project repo
    Path("test.jsonl"),
]
TEST_PATH = next((p for p in TEST_CANDIDATES if p.is_file()), None)

if TEST_PATH is None:
    # Try mounting Drive once and re-checking, in case it just isn't mounted yet.
    try:
        from google.colab import drive

        print("test.jsonl not found in /content — mounting Google Drive...")
        drive.mount("/content/drive")
        TEST_PATH = next((p for p in TEST_CANDIDATES if p.is_file()), None)
    except ImportError:
        pass  # not running on Colab; skip straight to the upload/manual-path fallback below

if TEST_PATH is None:
    # Last resort: interactive upload (Colab only). Outside Colab this leaves TEST_PATH unset
    # and the next cell raises a clear, actionable error instead of a silent wrong path.
    try:
        from google.colab import files

        print("Still not found. Opening a file-upload dialog — select your test.jsonl:")
        uploaded = files.upload()
        if uploaded:
            # files.upload() always saves into the current working directory, not /content
            # specifically — using Path.cwd() here matches its actual behaviour rather than
            # assuming a directory that may not exist on non-Colab Jupyter platforms.
            TEST_PATH = Path.cwd() / next(iter(uploaded))
    except ImportError:
        TEST_PATH = None

if TEST_PATH is None:
    raise FileNotFoundError(
        "Could not locate test.jsonl. Upload it to /content/, place it under "
        "/content/drive/MyDrive/, or set TEST_PATH = Path('/path/to/test.jsonl') manually "
        "in this cell and re-run."
    )

_slug = re.sub(r"[^a-z0-9]+", "-", MODEL_ID.lower()).strip("-")

# Output directory: prefer /content (real Colab — also what the optional Drive-save cell
# below assumes), otherwise write next to wherever TEST_PATH was actually found, since that
# directory is proven to exist and be readable in THIS environment. Path("w").open() does
# NOT create missing parent directories — writing straight to a hardcoded "/content/..." on
# a platform without a /content directory (e.g. Vertex AI Workbench, home at /home/jupyter)
# fails with FileNotFoundError, which is exactly the bug this replaces.
if Path("/content").is_dir():
    OUTPUT_DIR = Path("/content")
else:
    OUTPUT_DIR = TEST_PATH.parent
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # belt and braces regardless of which branch ran

RESULTS_JSONL = OUTPUT_DIR / f"{_slug}-eval.jsonl"
RESULTS_TXT = OUTPUT_DIR / f"{_slug}-results.txt"

# ---- These must match qa-finetuning_v6.ipynb exactly ----
NO_ANSWER = "මෙම ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත."
MAX_LENGTH = 2048
MAX_NEW_TOKENS = 48          # audited: covers 100% of gold answers under this tokenizer
REPETITION_PENALTY = 1.05
GROUNDING_THRESHOLD = 0.50
USE_GROUNDING = True

# ---- Context handling ----
# True  : always send the WHOLE provided context to the model, one generation per question.
# False : v6 behaviour — if a context exceeds the prompt budget, split it into overlapping
#         evidence windows, generate from the top TOP_K_WINDOWS, and keep the best-grounded
#         answer.
USE_FULL_CONTEXT = True
TOP_K_WINDOWS = 2            # only consulted when USE_FULL_CONTEXT is False

print("Model     :", MODEL_ID)
print("Test file :", TEST_PATH.resolve(), "| exists:", TEST_PATH.is_file())
print("Results   :", RESULTS_JSONL, "|", RESULTS_TXT)
print("Grounding :", f"on (threshold={GROUNDING_THRESHOLD}, top_k={TOP_K_WINDOWS})" if USE_GROUNDING else "off")

In [ ]:
# Only needed if the merged repo is private. Reads the token from a Colab secret first
# (key icon in the left sidebar -> add secret named HF_TOKEN), then an environment variable
# as a fallback for non-Colab Jupyter. Never hardcode a token in a cell.
from huggingface_hub import login


def _get_hf_token():
    try:
        from google.colab import userdata
        try:
            return userdata.get("HF_TOKEN")
        except Exception:
            return None
    except ImportError:
        return os.environ.get("HF_TOKEN")


_token = _get_hf_token()
if _token:
    login(token=_token, add_to_git_credential=False)
    print("Logged in to Hugging Face (Colab secret or HF_TOKEN environment variable).")
else:
    print("No HF_TOKEN found — continuing anonymously (fine for public repos).")
    print("For a private repo: click the key icon in the left sidebar, add a secret named")
    print("HF_TOKEN, enable notebook access, then re-run this cell.")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading merged model: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=model_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()
model.config.pad_token_id = tokenizer.pad_token_id

print("Loaded.  vocab:", len(tokenizer), "| dtype:", model_dtype, "| device:", model.device)
print("bos/eos/pad:", tokenizer.bos_token_id, "/", tokenizer.eos_token_id, "/", tokenizer.pad_token_id)

In [ ]:
# ---- Text handling, prompt, and data loading: copied from qa-finetuning_v6.ipynb ----

INSTRUCTION = f"""උපදෙස්: පහත සන්දර්භය පමණක් භාවිතා කර ප්‍රශ්නයට පිළිතුරු දෙන්න.
- පිළිතුර සන්දර්භයේ තිබේ නම්, එයින් කෙටිම නිශ්චිත වචන පෙළ පමණක් දෙන්න.
- අමතර පැහැදිලි කිරීම්, පිටත දැනුම හෝ අනුමාන එකතු නොකරන්න.
- සන්දර්භය ප්‍රශ්නයට අදාළ නොවේ නම්, ප්‍රශ්නයට පිළිතුරු දීමට සුදුසු නොවේ නම්, හෝ පිළිතුර සන්දර්භයේ පැහැදිලිව නොමැති නම්, හරියටම මෙය පමණක් දෙන්න: {NO_ANSWER}"""

SINHALA_WORD_RE = re.compile(r"[\w඀-෿]+", re.UNICODE)
STOPWORDS = {
    "හා", "සහ", "හෝ", "දී", "ද", "ය", "යි", "වේ", "විය", "වූ", "ලෙස",
    "විසින්", "සඳහා", "සිට", "දක්වා", "එම", "මෙම", "ඒ", "ඔහු", "ඇය",
    "කුමක්ද", "කවුද", "කවදාද", "කෙසේද", "කොපමණද", "මොනවාද",
}


def clean_text(value):
    text = unicodedata.normalize("NFC", str(value or ""))
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()


def lexical_tokens(value):
    tokens = [token.casefold() for token in SINHALA_WORD_RE.findall(clean_text(value))]
    return [token for token in tokens if len(token) >= 2 and token not in STOPWORDS]


def build_prompt(context, question):
    return (
        f"{INSTRUCTION}\n\n"
        f"සන්දර්භය:\n{clean_text(context)}\n\n"
        f"ප්‍රශ්නය:\n{clean_text(question)}\n\n"
        "පිළිතුර:\n"
    )


def canonical_answer(item):
    if item.get("answerable") is False:
        return NO_ANSWER
    answer = clean_text(item.get("answer", ""))
    return answer if answer else NO_ANSWER


def load_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f"Required JSONL file not found: {path}")

    records = []
    fingerprints = set()
    dropped = 0
    duplicates = 0

    with path.open("r", encoding="utf-8-sig") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"{path}:{line_number}: {error}") from error

            question = clean_text(item.get("question"))
            context = clean_text(item.get("context"))
            answerable = item.get("answerable")
            if type(answerable) is not bool:
                answerable = bool(clean_text(item.get("answer")))

            normalized = {
                "question": question,
                "context": context,
                "answer": clean_text(item.get("answer")),
                "answerable": answerable,
                "grade": item.get("grade"),
                "chapter": item.get("chapter"),
            }
            if not question or not context or (answerable and not normalized["answer"]):
                dropped += 1
                continue

            fingerprint = (
                normalized["question"],
                normalized["context"],
                normalized["answer"],
                normalized["answerable"],
            )
            if fingerprint in fingerprints:
                duplicates += 1
                continue
            fingerprints.add(fingerprint)
            records.append(normalized)

    return records, dropped, duplicates


print("Prompt and data helpers ready (v6-matched).")

In [ ]:
# ---- Evidence-window retrieval and grounded inference: copied from qa-finetuning_v6.ipynb ----

def token_supported(token, normalized_context):
    if token in normalized_context:
        return True
    # Sinhala case endings often add one character; a short stem check preserves grounded variants.
    return len(token) >= 4 and token[:-1] in normalized_context


def rank_context_windows(context, query, token_budget, top_k=1):
    context_ids = tokenizer(context, add_special_tokens=False)["input_ids"]
    if len(context_ids) <= token_budget:
        return [(context, 1.0)]

    query_tokens = set(lexical_tokens(query))
    stride = max(64, token_budget // 2)
    candidates = []

    for start in range(0, len(context_ids), stride):
        chunk_ids = context_ids[start : start + token_budget]
        if len(chunk_ids) < 32:
            continue
        chunk = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ).strip()
        normalized_chunk = " ".join(lexical_tokens(chunk))
        if query_tokens:
            matched = sum(token_supported(token, normalized_chunk) for token in query_tokens)
            score = matched / len(query_tokens)
        else:
            score = 0.0
        candidates.append((chunk, score, start))
        if start + token_budget >= len(context_ids):
            break

    candidates.sort(key=lambda value: (-value[1], value[2]))
    return [(chunk, score) for chunk, score, _ in candidates[:top_k]]


def context_budget(question, completion):
    fixed_tokens = len(tokenizer(build_prompt("", question), add_special_tokens=False)["input_ids"])
    completion_tokens = len(tokenizer(completion, add_special_tokens=False)["input_ids"])
    return max(128, MAX_LENGTH - fixed_tokens - completion_tokens - 24)


def evidence_support(answer, context):
    answer_tokens = lexical_tokens(answer)
    if not answer_tokens:
        return 0.0
    normalized_context = " ".join(lexical_tokens(context))
    supported = sum(token_supported(token, normalized_context) for token in answer_tokens)
    return supported / len(answer_tokens)


def generate_candidate(context_window, question):
    prompt = build_prompt(context_window, question)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=REPETITION_PENALTY,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[-1] :]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    answer = answer.splitlines()[0].strip(" []{}()<>\"'`") if answer else ""
    return answer


def run_qa(context, question, use_grounding=USE_GROUNDING):
    completion_stub = NO_ANSWER + (tokenizer.eos_token or "")
    budget = context_budget(question, completion_stub)

    if USE_FULL_CONTEXT:
        # Send the whole context, exactly as supplied — no chunking, one generation.
        windows = [(clean_text(context), 1.0)]
    else:
        windows = rank_context_windows(context, question, budget, top_k=TOP_K_WINDOWS)

    candidates = []

    for window, retrieval_score in windows:
        raw_answer = generate_candidate(window, question)
        support = 1.0 if is_no_answer(raw_answer) else evidence_support(raw_answer, window)
        candidates.append({
            "raw_answer": raw_answer,
            "window": window,
            "retrieval_score": retrieval_score,
            "support": support,
        })

    grounded = [
        candidate for candidate in candidates
        if candidate["raw_answer"] and not is_no_answer(candidate["raw_answer"])
        and candidate["support"] >= GROUNDING_THRESHOLD
    ]

    if grounded:
        best = max(
            grounded,
            key=lambda candidate: (candidate["support"], candidate["retrieval_score"]),
        )
        final_answer = best["raw_answer"]
    else:
        best = max(candidates, key=lambda candidate: candidate["retrieval_score"])
        final_answer = NO_ANSWER if use_grounding else best["raw_answer"]

    return {
        "answer": final_answer,
        "raw_answer": best["raw_answer"],
        "support": best["support"],
        "retrieval_score": best["retrieval_score"],
        "candidate_count": len(candidates),
    }


print("Grounded run_qa(context, question) ready (v6-matched).")

In [ ]:
# ---- Metrics: copied from qa-finetuning_v6.ipynb ----

def normalize_answer(value):
    text = clean_text(value).casefold()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" \t\r\n[]{}()<>\"'`.,!?;:।෴")


def is_no_answer(value):
    normalized = normalize_answer(value)
    return normalized == normalize_answer(NO_ANSWER) or "ප්‍රමාණවත් තොරතුරු නොමැත" in normalized


def token_f1(prediction, reference):
    prediction_digits = re.findall(r"\d+", normalize_answer(prediction))
    reference_digits = re.findall(r"\d+", normalize_answer(reference))
    # A wrong year is a wrong answer even when the surrounding words overlap.
    if reference_digits and prediction_digits != reference_digits:
        return 0.0
    prediction_tokens = lexical_tokens(prediction)
    reference_tokens = lexical_tokens(reference)
    if not prediction_tokens and not reference_tokens:
        return 1.0
    if not prediction_tokens or not reference_tokens:
        return 0.0
    common = Counter(prediction_tokens) & Counter(reference_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(prediction_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall)


def keyword_match(prediction, gold):
    """Legacy soft match, reported for continuity only."""
    pred, target = normalize_answer(prediction), normalize_answer(gold)
    if not pred or not target:
        return False
    if pred == target or target in pred or pred in target:
        return True
    target_tokens = [t for t in target.split() if len(t) > 1]
    if not target_tokens:
        return False
    return sum(1 for t in target_tokens if t in pred) / len(target_tokens) >= 0.6


print("Scoring helpers ready (v6-matched).")

In [ ]:
test_records, dropped_rows, duplicate_rows = load_jsonl(TEST_PATH)
LOAD_HEADER = [
    f"Loaded {len(test_records)} unique records from {TEST_PATH}",
    f"Dropped invalid/empty: {dropped_rows}; exact duplicates removed: {duplicate_rows}",
]
for line in LOAD_HEADER:
    print(line)
print(Counter(r["answerable"] for r in test_records))

gold_lengths = sorted(
    len(tokenizer(canonical_answer(r), add_special_tokens=False)["input_ids"]) for r in test_records
)
print(
    f"\nGold answer tokens: median {gold_lengths[len(gold_lengths)//2]}, "
    f"p95 {gold_lengths[int(len(gold_lengths)*0.95)]}, max {gold_lengths[-1]} "
    f"(MAX_NEW_TOKENS={MAX_NEW_TOKENS})"
)
if gold_lengths[-1] > MAX_NEW_TOKENS:
    print("WARNING: raise MAX_NEW_TOKENS — the longest gold answer does not fit.")

# Does every context fit in the prompt budget? In full-context mode an over-long context
# would be truncated by the tokenizer rather than windowed, silently hiding evidence — so
# report it rather than let it pass unnoticed.
_completion_stub = NO_ANSWER + (tokenizer.eos_token or "")
_ctx_tokens, _over_budget = [], []
for _r in test_records:
    _budget = context_budget(_r["question"], _completion_stub)
    _n = len(tokenizer(clean_text(_r["context"]), add_special_tokens=False)["input_ids"])
    _ctx_tokens.append(_n)
    if _n > _budget:
        _over_budget.append((_n, _budget))
_ctx_tokens.sort()
print(
    f"Context tokens    : median {_ctx_tokens[len(_ctx_tokens)//2]}, "
    f"p95 {_ctx_tokens[int(len(_ctx_tokens)*0.95)]}, max {_ctx_tokens[-1]} "
    f"(budget ~{context_budget(test_records[0]['question'], _completion_stub)})"
)
print(f"Context mode      : {'FULL context (no chunking)' if USE_FULL_CONTEXT else f'evidence windows, top-{TOP_K_WINDOWS}'}")
if _over_budget:
    print(
        f"WARNING: {len(_over_budget)}/{len(test_records)} contexts exceed the prompt budget "
        f"(largest {max(n for n, _ in _over_budget)} tokens). In full-context mode these will "
        "be truncated. Either raise MAX_LENGTH or set USE_FULL_CONTEXT = False to window them."
    )
else:
    print(f"All {len(test_records)} contexts fit the budget — nothing is truncated or dropped.")

In [ ]:
# ---- Prompt-format check + smoke test ----
row = test_records[0]
preview = build_prompt(row["context"], row["question"])

# Guard against silently drifting away from the trained format again.
assert preview.startswith("උපදෙස්: පහත සන්දර්භය පමණක්"), "instruction text does not match v6"
assert "\nසන්දර්භය:\n" in preview, "context label does not match v6"
assert "\nප්‍රශ්නය:\n" in preview, "question label does not match v6"
assert preview.endswith("පිළිතුර:\n"), "answer cue does not match v6 (must NOT end with '[')"
print("Prompt format matches the v6 training format.\n")
print("--- rendered prompt ---")
print(preview)

result = run_qa(row["context"], row["question"])
print("--- smoke test ---")
print("Question :", row["question"])
print("Reference:", canonical_answer(row))
print("Raw      :", result["raw_answer"])
print("Final    :", result["answer"])
print("Support  :", f"{result['support']:.3f}")

In [ ]:
# ---- Full evaluation ----
predictions = []
transcript_lines = list(LOAD_HEADER)

exact_correct = raw_exact_correct = 0
f1_total = 0.0
answerable_correct = answerable_total = 0
unanswerable_correct = unanswerable_total = 0
unsupported_rejections = 0
predicted_no_answer_count = correct_no_answer_count = 0
soft_correct = 0

with RESULTS_JSONL.open("w", encoding="utf-8", newline="\n") as results_file:
    for index, item in enumerate(test_records, 1):
        reference = canonical_answer(item)
        result = run_qa(item["context"], item["question"], use_grounding=USE_GROUNDING)
        prediction = result["answer"]
        raw_prediction = result["raw_answer"]

        exact = normalize_answer(prediction) == normalize_answer(reference)
        raw_exact = normalize_answer(raw_prediction) == normalize_answer(reference)
        f1 = token_f1(prediction, reference)
        predicted_no_answer = is_no_answer(prediction)

        exact_correct += int(exact)
        raw_exact_correct += int(raw_exact)
        f1_total += f1
        soft_correct += int(keyword_match(prediction, reference))
        predicted_no_answer_count += int(predicted_no_answer)
        correct_no_answer_count += int(predicted_no_answer and not item["answerable"])
        unsupported_rejections += int(
            prediction == NO_ANSWER and raw_prediction and not is_no_answer(raw_prediction)
        )

        if item["answerable"]:
            answerable_total += 1
            answerable_correct += int(exact)
        else:
            unanswerable_total += 1
            unanswerable_correct += int(exact)

        results_file.write(json.dumps({
            "index": index,
            "grade": item.get("grade"),
            "chapter": item.get("chapter"),
            "question": item["question"],
            "reference": reference,
            "raw_prediction": raw_prediction,
            "prediction": prediction,
            "answerable": item["answerable"],
            "exact_match": exact,
            "token_f1": f1,
            "evidence_support": result["support"],
            "retrieval_score": result["retrieval_score"],
        }, ensure_ascii=False) + "\n")
        results_file.flush()

        # The transcript FILE keeps the full v6-format block so it stays diffable against
        # llama_model_answers/*-v6-results.txt; the console shows only the fields worth
        # reading while the run streams.
        transcript_lines.extend([
            "",
            "=" * 100,
            f"[{index}/{len(test_records)}]",
            f"Grade    : {item.get('grade')} | Chapter: {item.get('chapter')}",
            f"Answerable: {item['answerable']}",
            f"Question : {item['question']}",
            f"Reference: {reference}",
            f"Raw      : {raw_prediction}",
            f"Final    : {prediction}",
            f"Support  : {result['support']:.3f}",
            f"Exact/F1 : {exact} / {f1:.3f}",
        ])

        print("\n".join([
            "",
            "=" * 100,
            f"[{index}/{len(test_records)}]",
            f"Answerable: {item['answerable']}",
            f"Question : {item['question']}",
            f"Expected : {reference}",
            f"Generated: {prediction}",
            f"Exact/F1 : {exact} / {f1:.3f}",
        ]), flush=True)

print("\nFinished evaluation.")

In [ ]:
total = len(test_records)
false_answers = unanswerable_total - unanswerable_correct
no_answer_precision = correct_no_answer_count / max(predicted_no_answer_count, 1)
no_answer_recall = correct_no_answer_count / max(unanswerable_total, 1)
no_answer_f1 = (
    2 * no_answer_precision * no_answer_recall / (no_answer_precision + no_answer_recall)
    if no_answer_precision + no_answer_recall else 0.0
)

SUMMARY = [
    "",
    "=" * 100,
    "EXTERNAL TEST RESULTS",
    "=" * 100,
    f"Model                : {MODEL_ID}",
    f"Test file            : {TEST_PATH}",
    f"Grounding gate       : {'on' if USE_GROUNDING else 'off'} "
    f"(threshold={GROUNDING_THRESHOLD}, top_k={TOP_K_WINDOWS})",
    "",
    f"Grounded exact match : {exact_correct}/{total} ({100 * exact_correct / total:.2f}%)",
    f"Raw exact match      : {raw_exact_correct}/{total} ({100 * raw_exact_correct / total:.2f}%)",
    f"Soft match (legacy)  : {soft_correct}/{total} ({100 * soft_correct / total:.2f}%)",
    f"Mean token F1        : {f1_total / total:.4f}",
    f"Answerable exact     : {answerable_correct}/{answerable_total} "
    f"({100 * answerable_correct / max(answerable_total, 1):.2f}%)",
    f"Unanswerable exact   : {unanswerable_correct}/{unanswerable_total} "
    f"({100 * unanswerable_correct / max(unanswerable_total, 1):.2f}%)",
    f"False-answer rate on unanswerable: {false_answers}/{unanswerable_total} "
    f"({100 * false_answers / max(unanswerable_total, 1):.2f}%)",
    f"No-answer precision/recall/F1: {no_answer_precision:.4f} / {no_answer_recall:.4f} / "
    f"{no_answer_f1:.4f}",
    f"Unsupported generations rejected: {unsupported_rejections}",
    "=" * 100,
]

for line in SUMMARY:
    print(line)
transcript_lines.extend(SUMMARY)

with RESULTS_TXT.open("w", encoding="utf-8", newline="\n") as handle:
    handle.write("\n".join(transcript_lines) + "\n")

print(f"\nPer-item results : {RESULTS_JSONL}")
print(f"Transcript       : {RESULTS_TXT}")
print("\nThis transcript uses the same block format as llama_model_answers/*-v6-results.txt")
print("and the Modal notebook's output, so it can be diffed directly against either.")

In [ ]:
# ---- Optional: copy results to Google Drive, if mounted ----
DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/sinhala-qa-results")

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    for src in (RESULTS_JSONL, RESULTS_TXT):
        dest = DRIVE_SAVE_DIR / src.name
        dest.write_bytes(src.read_bytes())
        print("Copied to Drive:", dest)
else:
    print("Google Drive is not mounted — results remain only in /content "
          "(this Colab session's storage, cleared when the runtime recycles).")
    print("To keep them: run `from google.colab import drive; drive.mount('/content/drive')`")
    print("then re-run this cell, or download the files from the Colab file browser.")

In [ ]:
# ---- All QA pairs: question, expected answer, generated answer ----
# Reads back the saved per-item results, so this can be re-run on its own without
# repeating inference.
all_rows = [json.loads(l) for l in RESULTS_JSONL.open(encoding="utf-8") if l.strip()]

for r in all_rows:
    print("=" * 100)
    print(f"[{r['index']}/{len(all_rows)}]")
    print("Answerable:", r["answerable"])
    print("Question :", r["question"])
    print("Expected :", r["reference"])
    print("Generated:", r["prediction"])
    print(f"Exact/F1 : {r['exact_match']} / {r['token_f1']:.3f}")

In [ ]:
# ---- Failure inspection ----
rows = [json.loads(l) for l in RESULTS_JSONL.open(encoding="utf-8") if l.strip()]

hallucinated = [r for r in rows if not r["answerable"] and not is_no_answer(r["prediction"])]
print(f"Fabricated answers on unanswerable questions: {len(hallucinated)}")
for r in hallucinated[:5]:
    print("-" * 90)
    print("Q   :", r["question"])
    print("Pred:", r["prediction"], f"| support {r['evidence_support']:.3f}")

low_f1 = [r for r in rows if r["answerable"] and r["token_f1"] < 0.5]
print(f"\nAnswerable rows below 0.5 token F1: {len(low_f1)}")
for r in low_f1[:10]:
    print("-" * 90)
    print("Q   :", r["question"])
    print("Ref :", r["reference"])
    print("Raw :", r["raw_prediction"])
    print("Pred:", r["prediction"], f"| F1 {r['token_f1']:.3f} | support {r['evidence_support']:.3f}")